In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


In [18]:
month_date = 202508
channel = ['零售','工程','电商']
current_date = pd.Timestamp('2025-08-31')

productgroup_map={
    '吸油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜': ['消毒柜'],
    '热水器': ['热水器','两用炉'],
    '净水机': ['家用净水机','商用净水机'],
    '洗碗机': ['水槽洗碗机','嵌入式洗碗机']
}

In [19]:
df = pd.read_excel(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾.xlsx')
df['物料号'] = df['商品编码'].astype(str).map(lambda x: x[:13])
print(len(df))
df.head()


306178


,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外,物料号
0,1009001100002,工程,1,蒸烤烹饪机,3550,3550,ZK50-01-F1.i,国内,1009001100002
1,1001002100022,工程,2,吸油烟机,1508,3016,JC03A,国内,1001002100022
2,1002003400032,工程,2,灶具,900,1800,TH3B,国内,1002003400032
3,1001002000018,工程,1,吸油烟机,3668,3668,03-X1A,国内,1001002000018
4,1003000500029,工程,1,消毒柜,1550,1550,ZTD100J-J31,国内,1003000500029


In [20]:
# 长尾只看3大渠道。每个渠道的停止销售时间和既定时间的差距，所以只需要保留物料号、渠道、产品组、标准型号、国内/海外
df1 = df.copy()
df1 = df1[df1['渠道'].isin(channel)]
df1 = df1[['物料号','渠道','产品组','标准型号','国内/海外']].drop_duplicates().reset_index(drop=True)
df1

,物料号,渠道,产品组,标准型号,国内/海外
0,1009001100002,工程,蒸烤烹饪机,ZK50-01-F1.i,国内
1,1001002100022,工程,吸油烟机,JC03A,国内
2,1002003400032,工程,灶具,TH3B,国内
3,1001002000018,工程,吸油烟机,03-X1A,国内
4,1003000500029,工程,消毒柜,ZTD100J-J31,国内
...,...,...,...,...,...
1859,1009000900029,电商,灶蒸烤烹饪机,JZT-ZK60-03-X5.i,国内
1860,1009000900001,零售,灶蒸烤烹饪机,JZT-ZK60-X3.i,国内
1861,1009000900001,电商,灶蒸烤烹饪机,JZT-ZK60-X3.i,国内
1862,1009000600004,电商,蒸烤烹饪机,ZK-T1,国内


In [21]:
df_product_life = pd.read_excel(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\产品生命周期状态全表20250703.xlsx')
# 转换物料号列为字符串类型，并只取前13位
df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).map(lambda x: x[:13])
df_product_life = df_product_life[df_product_life['物料号'].str.len()>10]
df_product_life['渠道'] = df_product_life['下属渠道']
#####
df_product_life_map_df = df_product_life[['物料号','渠道','对应渠道状态','产品状态','产品型号','停止销售时间']]
df_product_life_map_df 

,物料号,渠道,对应渠道状态,产品状态,产品型号,停止销售时间
4,1004000200090,NaN,NaN,停止发货,10T-JSG15-0606FR,NaT
5,1004000200079,NaN,NaN,停止发货,10T-JSG19-0607FR,NaT
6,1004000200084,NaN,NaN,停止发货,10T-JSG25-0608FR,NaT
7,1004000200035,NaN,NaN,停止发货,10T-JSQ16-0601,NaT
8,1004000200060,NaN,NaN,停止发货,10T-JSQ16-0601FR,NaT
...,...,...,...,...,...,...
8928,1001002200009,工程,在售,量产,iMES-45-C1,NaT
8929,1001002200005,工程,在售,量产,iMES-60-D1,NaT
8930,1001002200001,NaN,NaN,作废,iMES-C1,NaT
8931,1001002200007,工程,在售,量产,iMES-D1G-K1,NaT


In [28]:
df_calu = pd.merge(df1,df_product_life_map_df,how='left',on=['物料号','渠道'])
df_calu['停止销售时间'] = pd.to_datetime(df_calu['停止销售时间'])
df_calu = df_calu[df_calu['渠道'].isin(channel)]
df_calu = df_calu[df_calu['产品状态'].isin(['停止销售','停止生产'])]
df_calu

,物料号,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间
8,1002001500070,工程,灶具,FZ6G,国内,停止销售,停止销售,JZT-FZ6G-12T,2024-10-31
9,1001000800337,工程,吸油烟机,EH37,国内,停止销售,停止销售,CXW-258-EH37,2022-07-07
11,1001002000004,工程,吸油烟机,X1A,国内,停止销售,停止销售,CXW-258-X1A,2025-06-19
14,1009000600012,工程,蒸烤烹饪机,ZK-TS1.i,国内,停止销售,停止销售,ZK-TS1.i,2025-04-25
18,1018000200002,工程,嵌入式洗碗机,JPCD11E-NT02,国内,停止销售,停止销售,JPCD11E-NT02,2024-07-03
...,...,...,...,...,...,...,...,...,...
1855,1007000400001,工程,蒸箱,SCD42-F1,国内,停止发货,停止销售,SCD42-F1,2025-01-19
1856,1008000200075,零售,水槽洗碗机,JBSD2F-Q5S,国内,停止销售,停止销售,JBSD2F-Q5S(不带底),2020-10-30
1857,1008000400008,零售,水槽洗碗机,JPSD2T-G3,国内,停止销售,停止销售,JPSD2T-GD03,2023-01-12
1858,1008000400007,零售,水槽洗碗机,JPSD2T-G3L,国内,停止销售,停止销售,JPSD2T-GD03L,2023-01-12


In [29]:
from calendar import month
from pandas import DateOffset
#  只要有一个产品型号是长尾，那么这个标准型号就是长尾
for index,row in df_calu.iterrows():
    if row['渠道'] == '零售':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df_calu.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '电商':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df_calu.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '工程':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=30):
            df_calu.loc[index,'是否长尾型号'] = '是'
df_calu['是否长尾型号'] = df_calu['是否长尾型号'].fillna('否')
df_calu

,物料号,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间,是否长尾型号
8,1002001500070,工程,灶具,FZ6G,国内,停止销售,停止销售,JZT-FZ6G-12T,2024-10-31,否
9,1001000800337,工程,吸油烟机,EH37,国内,停止销售,停止销售,CXW-258-EH37,2022-07-07,是
11,1001002000004,工程,吸油烟机,X1A,国内,停止销售,停止销售,CXW-258-X1A,2025-06-19,否
14,1009000600012,工程,蒸烤烹饪机,ZK-TS1.i,国内,停止销售,停止销售,ZK-TS1.i,2025-04-25,否
18,1018000200002,工程,嵌入式洗碗机,JPCD11E-NT02,国内,停止销售,停止销售,JPCD11E-NT02,2024-07-03,否
...,...,...,...,...,...,...,...,...,...,...
1855,1007000400001,工程,蒸箱,SCD42-F1,国内,停止发货,停止销售,SCD42-F1,2025-01-19,否
1856,1008000200075,零售,水槽洗碗机,JBSD2F-Q5S,国内,停止销售,停止销售,JBSD2F-Q5S(不带底),2020-10-30,是
1857,1008000400008,零售,水槽洗碗机,JPSD2T-G3,国内,停止销售,停止销售,JPSD2T-GD03,2023-01-12,是
1858,1008000400007,零售,水槽洗碗机,JPSD2T-G3L,国内,停止销售,停止销售,JPSD2T-GD03L,2023-01-12,是


In [37]:
df_out = pd.DataFrame()
df_out['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')]['标准型号'].nunique()
    df_out.loc[df_out['产品类别']==k,'长尾标准型号数量'] = vals
    vals = df_calu[(df_calu['产品组'].isin(v))]['标准型号'].nunique()
    df_out.loc[df_out['产品类别']==k,'标准型号数量'] = vals
df_out['长尾标准型号占比'] = df_out['长尾标准型号数量']/df_out['标准型号数量']
df_out
df_out1 = pd.DataFrame()
df_out1['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'零售长尾产品型号数量'] = vals
    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'零售产品型号数量'] = vals

    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='电商')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'电商长尾产品型号数量'] = vals
    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='电商')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'电商产品型号数量'] = vals

    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='工程')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'工程长尾产品型号数量'] = vals
    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'工程产品型号数量'] = vals

    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'全渠道长尾产品型号数量'] = vals
    vals = df_calu[(df_calu['产品组'].isin(v))]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'全渠道产品型号数量'] = vals
df_out1['零售长尾产品型号占比'] = df_out1['零售长尾产品型号数量']/df_out1['零售产品型号数量']
df_out1['电商长尾产品型号占比'] = df_out1['电商长尾产品型号数量']/df_out1['电商产品型号数量']
df_out1['工程长尾产品型号占比'] = df_out1['工程长尾产品型号数量']/df_out1['工程产品型号数量']
df_out1['全渠道长尾产品型号占比'] = df_out1['全渠道长尾产品型号数量']/df_out1['全渠道产品型号数量']
df_out1


,产品类别,零售长尾产品型号数量,零售产品型号数量,电商长尾产品型号数量,电商产品型号数量,工程长尾产品型号数量,工程产品型号数量,全渠道长尾产品型号数量,全渠道产品型号数量,零售长尾产品型号占比,电商长尾产品型号占比,工程长尾产品型号占比,全渠道长尾产品型号占比
0,吸油烟机,16.0,31.0,11.0,24.0,24.0,31.0,40.0,71.0,0.516129,0.458333,0.774194,0.563380
1,灶具,18.0,35.0,3.0,11.0,5.0,35.0,23.0,62.0,0.514286,0.272727,0.142857,0.370968
2,蒸烤微合计,7.0,19.0,5.0,21.0,0.0,19.0,8.0,25.0,0.368421,0.238095,0.000000,0.320000
3,灶集成,3.0,11.0,2.0,7.0,0.0,11.0,3.0,11.0,0.272727,0.285714,0.000000,0.272727
4,消毒柜,1.0,2.0,0.0,1.0,0.0,2.0,1.0,4.0,0.500000,0.000000,0.000000,0.250000
5,热水器,5.0,5.0,0.0,0.0,1.0,5.0,6.0,7.0,1.000000,NaN,0.200000,0.857143
6,净水机,3.0,11.0,1.0,8.0,0.0,11.0,3.0,11.0,0.272727,0.125000,0.000000,0.272727
7,洗碗机,10.0,14.0,5.0,7.0,4.0,14.0,13.0,23.0,0.714286,0.714286,0.285714,0.565217


In [38]:
with pd.ExcelWriter(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\长尾统计结果.xlsx') as writer:
    df_out.to_excel(writer,sheet_name='标准型号统计',index=False)
    df_out1.to_excel(writer,sheet_name='分渠道产品型号统计',index=False)
